# Milestone 4.33 - Missing Value Detection in Pandas

This notebook demonstrates comprehensive detection of missing values in Pandas DataFrames:

- Load a dataset into a Pandas DataFrame
- Detect missing values in dataset
- Summarize missing values by column and/or row
- Inspect rows containing missing data
- Clean and readable code

**Note**: No data cleaning or imputation is performed - only detection and analysis.

In [ ]:
import pandas as pd
import numpy as np

# Create a healthcare dataset with intentional missing values
np.random.seed(123)  # For reproducible results

# Generate base patient data
num_patients = 50
patient_ids = range(2001, 2001 + num_patients)

# Create base data
data = {
    'patient_id': patient_ids,
    'name': [f'Patient_{i}' for i in range(1, num_patients + 1)],
    'age': np.random.randint(18, 85, num_patients),
    'gender': np.random.choice(['Male', 'Female'], num_patients),
    'blood_pressure_systolic': np.random.randint(90, 180, num_patients),
    'blood_pressure_diastolic': np.random.randint(60, 120, num_patients),
    'heart_rate': np.random.randint(50, 120, num_patients),
    'temperature': np.round(np.random.uniform(96.0, 101.0, num_patients), 1),
    'department': np.random.choice(['Cardiology', 'Neurology', 'Orthopedics', 'Emergency'], num_patients),
    'admission_date': pd.date_range('2024-01-01', periods=num_patients, freq='6H'),
    'treatment': np.random.choice(['Surgery', 'Medication', 'Therapy', 'Observation'], num_patients),
    'insurance_type': np.random.choice(['Private', 'Medicare', 'Medicaid', 'Self-Pay'], num_patients),
    'total_cost': np.round(np.random.uniform(1000, 50000, num_patients), 2)
}

# Create DataFrame
df = pd.DataFrame(data)

# Intentionally introduce missing values to demonstrate detection
# Scenario 1: Missing vital signs for some patients
missing_vitals_indices = np.random.choice(df.index, size=8, replace=False)
df.loc[missing_vitals_indices[:4], 'heart_rate'] = np.nan
df.loc[missing_vitals_indices[4:6], 'temperature'] = np.nan
df.loc[missing_vitals_indices[6:8], 'blood_pressure_systolic'] = np.nan

# Scenario 2: Missing administrative data
missing_admin_indices = np.random.choice(df.index, size=6, replace=False)
df.loc[missing_admin_indices[:3], 'insurance_type'] = np.nan
df.loc[missing_admin_indices[3:], 'treatment'] = np.nan

# Scenario 3: Missing cost data for some patients
missing_cost_indices = np.random.choice(df.index, size=4, replace=False)
df.loc[missing_cost_indices, 'total_cost'] = np.nan

# Scenario 4: Multiple missing values in same rows
multi_missing_indices = np.random.choice(df.index, size=3, replace=False)
for idx in multi_missing_indices:
    df.loc[idx, ['blood_pressure_diastolic', 'department', 'admission_date']] = np.nan

print("Healthcare DataFrame with missing values loaded successfully!")
print(f"Dataset created with {len(df)} patient records")
print(f"Total missing values introduced: {df.isnull().sum().sum()}")

## 1. Missing Value Detection Overview

Using multiple methods to detect missing values in the DataFrame.

In [ ]:
# Method 1: Basic missing value detection using isnull()
print("=== METHOD 1: Basic Detection with isnull() ===")
missing_mask = df.isnull()
print("Missing value mask (True = missing, False = present):")
print(missing_mask.head(10))

# Method 2: Count total missing values
print("\n=== METHOD 2: Total Missing Values ===")
total_missing = df.isnull().sum().sum()
total_cells = df.shape[0] * df.shape[1]
missing_percentage = (total_missing / total_cells) * 100
print(f"Total missing values: {total_missing}")
print(f"Total data cells: {total_cells}")
print(f"Missing percentage: {missing_percentage:.2f}%")

# Method 3: Boolean detection for any missing values
print("\n=== METHOD 3: Boolean Detection ===")
has_missing = df.isnull().any().any()
print(f"DataFrame has missing values: {has_missing}")
print(f"Columns with missing values: {df.isnull().any().sum()}")
print(f"Rows with missing values: {df.isnull().any(axis=1).sum()}")

## 2. Column-wise Missing Value Summary

Analyzing missing values by column to identify data quality issues.

In [ ]:
# Column-wise missing value analysis
print("=== COLUMN-WISE MISSING VALUE ANALYSIS ===")
missing_by_column = df.isnull().sum()
missing_percentage_by_column = (missing_by_column / len(df)) * 100

# Create summary DataFrame
column_summary = pd.DataFrame({
    'Missing_Count': missing_by_column,
    'Missing_Percentage': missing_percentage_by_column.round(2),
    'Data_Type': df.dtypes,
    'Non_Missing_Count': df.count()
})

# Sort by missing count (descending)
column_summary = column_summary.sort_values('Missing_Count', ascending=False)

print("Missing Value Summary by Column:")
print(column_summary)

# Identify columns with and without missing values
print("\n=== COLUMN CLASSIFICATION ===")
columns_with_missing = column_summary[column_summary['Missing_Count'] > 0]
columns_without_missing = column_summary[column_summary['Missing_Count'] == 0]

print(f"\nColumns WITH missing values ({len(columns_with_missing)}):")
for col in columns_with_missing.index:
    count = columns_with_missing.loc[col, 'Missing_Count']
    percentage = columns_with_missing.loc[col, 'Missing_Percentage']
    print(f"  • {col}: {count} missing ({percentage:.1f}%)")

print(f"\nColumns WITHOUT missing values ({len(columns_without_missing)}):")
for col in columns_without_missing.index:
    print(f"  • {col}: Complete data")

# Data type analysis of missing values
print("\n=== MISSING VALUES BY DATA TYPE ===")
missing_by_type = df.isnull().sum().groupby(df.dtypes).sum()
for dtype, missing_count in missing_by_type.items():
    if missing_count > 0:
        print(f"  • {dtype}: {missing_count} missing values")

## 3. Row-wise Missing Value Analysis

Examining rows to identify patterns of missing data.

In [ ]:
# Row-wise missing value analysis
print("=== ROW-WISE MISSING VALUE ANALYSIS ===")

# Count missing values per row
missing_by_row = df.isnull().sum(axis=1)
row_summary = pd.DataFrame({
    'Missing_Count': missing_by_row,
    'Missing_Percentage': (missing_by_row / df.shape[1] * 100).round(2)
})

# Add patient identifiers for context
row_summary['patient_id'] = df['patient_id']
row_summary['patient_name'] = df['name']

print("Missing Value Summary by Row (First 15):")
print(row_summary[['patient_id', 'patient_name', 'Missing_Count', 'Missing_Percentage']].head(15))

# Analyze missing value patterns
print("\n=== ROW MISSING VALUE PATTERNS ===")
rows_no_missing = (missing_by_row == 0).sum()
rows_some_missing = ((missing_by_row > 0) & (missing_by_row < df.shape[1])).sum()
rows_many_missing = (missing_by_row >= df.shape[1] * 0.5).sum()  # 50% or more missing

print(f"Rows with NO missing values: {rows_no_missing} ({(rows_no_missing/len(df))*100:.1f}%)")
print(f"Rows with SOME missing values: {rows_some_missing} ({(rows_some_missing/len(df))*100:.1f}%)")
print(f"Rows with MANY missing values (≥50%): {rows_many_missing} ({(rows_many_missing/len(df))*100:.1f}%)")

# Find rows with maximum missing values
max_missing_row = missing_by_row.idxmax()
max_missing_count = missing_by_row.max()
print(f"\nRow with most missing values: Patient {df.loc[max_missing_row, 'name']} (ID: {df.loc[max_missing_row, 'patient_id']})")
print(f"Missing values in this row: {max_missing_count} out of {df.shape[1]} columns ({(max_missing_count/df.shape[1])*100:.1f}%)")

## 4. Inspection of Rows with Missing Values

Detailed examination of specific rows containing missing data.

In [ ]:
# Get all rows with missing values
rows_with_missing = df[df.isnull().any(axis=1)]

print("=== DETAILED INSPECTION OF ROWS WITH MISSING VALUES ===")
print(f"Total rows with missing values: {len(rows_with_missing)}")
print(f"Percentage of rows with missing values: {(len(rows_with_missing)/len(df))*100:.1f}%")

# Show rows with missing values, highlighting missing data
print("\nRows with missing values (NaN indicates missing data):")
print("=" * 80)

for idx, row in rows_with_missing.iterrows():
    missing_cols = df.columns[df.isnull().loc[idx]].tolist()
    print(f"\nRow {idx} - Patient: {row['name']} (ID: {row['patient_id']})")
    print(f"Missing columns ({len(missing_cols)}): {', '.join(missing_cols)}")
    
    # Show available data for this row
    available_data = row.dropna().to_dict()
    print("Available data:")
    for key, value in available_data.items():
        print(f"  • {key}: {value}")

# Group rows by number of missing values
print("\n=== ROWS GROUPED BY MISSING VALUE COUNT ===")
missing_counts = missing_by_row.value_counts().sort_index()
for missing_count, row_count in missing_counts.items():
    if missing_count > 0:  # Only show rows with missing values
        percentage = (row_count / len(df)) * 100
        print(f"{missing_count} missing values: {row_count} rows ({percentage:.1f}%)")

## 5. Missing Value Pattern Analysis

Analyzing patterns and relationships in missing data.

In [ ]:
# Analyze missing value patterns
print("=== MISSING VALUE PATTERN ANALYSIS ===")

# Create missing value correlation matrix
missing_corr = df.isnull().corr()
print("Missing Value Correlation Matrix:")
print(missing_corr.round(2))

# Find pairs of columns that tend to be missing together
print("\n=== COLUMNS THAT TEND TO BE MISSING TOGETHER ===")
missing_pairs = []
for i, col1 in enumerate(df.columns):
    for j, col2 in enumerate(df.columns):
        if i < j:  # Avoid duplicates
            # Calculate how often both are missing together
            both_missing = ((df[col1].isnull()) & (df[col2].isnull())).sum()
            if both_missing > 0:
                missing_pairs.append((col1, col2, both_missing))

# Sort by frequency
missing_pairs.sort(key=lambda x: x[2], reverse=True)

for col1, col2, count in missing_pairs[:10]:  # Show top 10
    print(f"  • {col1} + {col2}: {count} rows missing both")

# Missing value heatmap summary
print("\n=== MISSING VALUE HEATMAP SUMMARY ===")
missing_matrix = df.isnull().sum().to_frame('Missing_Count')
missing_matrix['Missing_Percentage'] = (missing_matrix['Missing_Count'] / len(df) * 100).round(1)
missing_matrix = missing_matrix.sort_values('Missing_Count', ascending=False)

print("Columns sorted by missing value frequency:")
for col, data in missing_matrix.iterrows():
    if data['Missing_Count'] > 0:
        bar_length = int(data['Missing_Percentage'] / 2)  # Scale for display
        bar = '█' * bar_length + '░' * (50 - bar_length)
        print(f"  {col:20} |{bar}| {data['Missing_Count']:3d} ({data['Missing_Percentage']:5.1f}%)")

## 6. Summary and Findings

Comprehensive summary of missing value detection results.

In [ ]:
# Complete missing value analysis summary
print("=== COMPREHENSIVE MISSING VALUE ANALYSIS SUMMARY ===")
print("=" * 60)

# Overall statistics
total_missing = df.isnull().sum().sum()
total_cells = df.shape[0] * df.shape[1]
overall_missing_percentage = (total_missing / total_cells) * 100
columns_with_missing = (df.isnull().sum() > 0).sum()
rows_with_missing = (df.isnull().any(axis=1)).sum()

print(f"📊 OVERALL MISSING VALUE STATISTICS:")
print(f"   • Total missing values: {total_missing}")
print(f"   • Total data cells: {total_cells}")
print(f"   • Overall missing percentage: {overall_missing_percentage:.2f}%")
print(f"   • Columns with missing values: {columns_with_missing}/{df.shape[1]}")
print(f"   • Rows with missing values: {rows_with_missing}/{df.shape[0]}")

# Column-specific findings
print(f"\n📋 COLUMN-SPECIFIC FINDINGS:")
missing_by_col = df.isnull().sum()
most_missing_col = missing_by_col.idxmax()
most_missing_count = missing_by_col.max()

print(f"   • Most affected column: {most_missing_col} ({most_missing_count} missing values)")
print(f"   • Least affected columns: {missing_by_col[missing_by_col == missing_by_col.min()].index.tolist()}")

# Row-specific findings
print(f"\n👥 ROW-SPECIFIC FINDINGS:")
missing_by_row = df.isnull().sum(axis=1)
max_missing_row_idx = missing_by_row.idxmax()
max_missing_row_count = missing_by_row.max()

print(f"   • Row with most missing values: Patient {df.loc[max_missing_row_idx, 'name']} (ID: {df.loc[max_missing_row_idx, 'patient_id']})")
print(f"   • Maximum missing values in a row: {max_missing_row_count}/{df.shape[1]}")
print(f"   • Complete rows (no missing values): {(missing_by_row == 0).sum()}")

# Data quality assessment
print(f"\n✅ DATA QUALITY ASSESSMENT:")
if overall_missing_percentage < 5:
    print(f"   • Data quality: EXCELLENT (< 5% missing)")
elif overall_missing_percentage < 10:
    print(f"   • Data quality: GOOD (5-10% missing)")
elif overall_missing_percentage < 20:
    print(f"   • Data quality: FAIR (10-20% missing)")
else:
    print(f"   • Data quality: POOR (> 20% missing)")

print(f"\n🎯 KEY INSIGHTS:")
print(f"   • Missing values are {'concentrated' if columns_with_missing < df.shape[1] * 0.5 else 'distributed'} across columns")
print(f"   • Missing values are {'concentrated' if rows_with_missing < df.shape[0] * 0.5 else 'distributed'} across rows")
print(f"   • Dataset {'requires attention' if overall_missing_percentage > 10 else 'is manageable'} for data cleaning")

print(f"\n🔍 DETECTION COMPLETE:")
print(f"   • All missing values successfully detected and analyzed")
print(f"   • No data modification or imputation performed")
print(f"   • Ready for data cleaning decisions based on this analysis")